In [ ]:
import pandas as pd

# Loading in data and renaming columns
def load_data(file_path):
    df = pd.read_csv(file_path, header=None, sep="\t")

    df = df.rename(columns={
        0: "id",
        1: "modified",
        2: "sequence",
        3: "position"
    })
    df["position"] = df["position"].astype("Int64")
    return df



file = "/home/callum/projects/sno_analysis/data/raw/SNO_combined.txt"
df = load_data(file)

print("duplicates...")
print(df.duplicated().sum())


print(df.dtypes)

# function exploring dataset and giving basic info and counts
# def explore_data(df):
#     """Print a summary of shape, dtypes, nulls, and key value counts."""
#     print("Shape:", df.shape)
#     print("\nDtypes:\n", df.dtypes)
#     print("\nMissing values per column:\n", df.isnull().sum())
#     print("\nUnique proteins:", df["id"].nunique())
#     print(f"\nClass balance (modified): {df['modified'].value_counts()}\n")
# explore_data(df)


# --- Identifying conflicting sites ---
# for each id and position (each unique site), how many unique modified values are there?
# ie are there both positives and negatives for the same site
site_counts = df.groupby(["id", "position"])["modified"].nunique()
# this gives a series with id and position as index and 1 column - the number of unique entries

# filter for values greater than 1 - ie 2 different modification values present for the same site
conflicting_sites = site_counts[site_counts > 1]
print(conflicting_sites)

# reset the index to make id and position columns in a proper dataframe
conflicting_sites_df = conflicting_sites.reset_index()

# rename the count column from this df to identify column that marks conflicts
conflicting_sites_df = conflicting_sites_df.rename(columns={"modified": "conflicts"})
print(conflicting_sites_df)


# --- Merging conflict info back onto the full dataset ---
print("merging dataframes...")
merged = pd.merge(df, conflicting_sites_df, on=["id", "position"], how="left", indicator=True)

# confirming number of matched rows
# there are 334 - conflicting pairs have an entry for both the 0 and the 1 modification
print(merged["_merge"].value_counts())


# --- Filtering: keep non-conflicting rows, plus only the positive row for conflicts ---
before = int(merged.shape[0])

condition1 = merged["_merge"] == "left_only"
condition2 = (merged["_merge"] == "both") & (merged["modified"] == 1)
merged_filtered = merged[condition1 | condition2]

after = int(merged_filtered.shape[0])
print(f"Rows dropped: {before - after}")

# drop the helper columns now that filtering is done
to_drop = ["_merge", "conflicts"]
merged_filtered_clean = merged_filtered.drop(to_drop, axis="columns")

print(merged_filtered_clean.head())
print(merged_filtered_clean.shape[0])



duplicates...
0
id            str
modified    int64
sequence      str
position    Int64
dtype: object
id      position
A2ASS6  13296       2
        14760       2
        18679       2
        21561       2
        21780       2
                   ..
Q9WUM4  456         2
Q9WV60  14          2
        317         2
Q9Z0E0  415         2
        725         2
Name: modified, Length: 167, dtype: int64
         id  position  conflicts
0    A2ASS6     13296          2
1    A2ASS6     14760          2
2    A2ASS6     18679          2
3    A2ASS6     21561          2
4    A2ASS6     21780          2
..      ...       ...        ...
162  Q9WUM4       456          2
163  Q9WV60        14          2
164  Q9WV60       317          2
165  Q9Z0E0       415          2
166  Q9Z0E0       725          2

[167 rows x 3 columns]
merging dataframes...
_merge
left_only     23897
both            334
right_only        0
Name: count, dtype: int64
Rows dropped: 167
       id  modified                         

# made into a function


In [ ]:
def resolve_label_conflicts(df, group_cols, label_col, keep_value=1):
    """
    For rows sharing the same group_cols (e.g. id + position), where label_col
    has more than one distinct value, keep only the row where label_col == keep_value.
    Rows with no conflict are left untouched.
    """
    # count distinct label values per group
    label_counts = df.groupby(group_cols)[label_col].nunique()
    conflicting_groups = label_counts[label_counts > 1].reset_index()
    conflicting_groups = conflicting_groups.rename(columns={label_col: "n_unique_labels"})

    # flag which rows belong to a conflicting group
    merged = pd.merge(df, conflicting_groups, on=group_cols, how="left", indicator=True)

    # keep: non-conflicting rows, plus only the "keep_value" row within conflicting groups
    no_conflict = merged["_merge"] == "left_only"
    resolved_conflict = (merged["_merge"] == "both") & (merged[label_col] == keep_value)

    result = merged[no_conflict | resolved_conflict].drop(columns=["_merge", "n_unique_labels"])
    return result.reset_index(drop=True)

In [ ]:
# filtering for sites

print(merged_filtered_clean.head())


